# Linguistic model — what verbs & tenses did it learn?

The `linguistic` model uses spaCy to extract verbs between the two entities, their tense, negation, distance, and entity ordering — plus a **learned bag-of-verb-lemmas** — and feeds these into an XGBoost classifier (one head per target: `at` and `isAt`). There are **no hand-written verb lists**: the classifier learns which verbs matter directly from the labels.

This notebook inspects the resulting feature importances to see what the model actually learned.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
import matplotlib
matplotlib.use("Agg")

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from hipe.data.pairs import load_pairs
from hipe.models.linguistic import LinguisticModel

DATA_ROOT = Path("/Users/mayadeneva/Documents/uni/ozt/hipe/data/raw/HIPE-2026-data/data/sandbox")

train = []
for lang in ("en", "de", "fr"):
    train.extend(load_pairs(DATA_ROOT / f"{lang}-train.jsonl"))

print(f"Total training pairs: {len(train)}")

m = LinguisticModel()
m.fit(train)

print(f"Total features after fitting: {len(m.vec.get_feature_names_out())}")

Total training pairs: 6170


In [ ]:
feature_names = m.vec.get_feature_names_out()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, head, title_tag in zip(axes, [m._at, m._isat], ["at", "isAt"]):
    importances = head.clf.feature_importances_
    top_idx = np.argsort(importances)[::-1][:20]
    top_names = feature_names[top_idx]
    top_vals = importances[top_idx]

    # sort ascending for horizontal bar (most important at top)
    order = np.argsort(top_vals)
    ax.barh(top_names[order], top_vals[order], color="steelblue")
    ax.set_title(f"Top features — {title_tag}", fontsize=13)
    ax.set_xlabel("Feature importance")
    ax.tick_params(axis="y", labelsize=9)

plt.tight_layout()
plt.savefig("/Users/mayadeneva/Documents/uni/ozt/hipe/notebooks/02_top_features.png", dpi=120)
plt.show()
print("Saved 02_top_features.png")

# Print top-5 for each target
for head, tag in [(m._at, "at"), (m._isat, "isAt")]:
    imp = head.clf.feature_importances_
    top5 = np.argsort(imp)[::-1][:5]
    print(f"\nTop 5 for {tag}:")
    for i in top5:
        print(f"  {feature_names[i]}: {imp[i]:.4f}")

Saved 02_top_features.png

Top 5 for at:
  vb_want: 0.0058
  vb_reconnaître: 0.0052
  vb_naître: 0.0048
  vb_accompagner: 0.0048
  vb_know: 0.0046

Top 5 for isAt:
  has_pres: 0.0117
  vb_accompagner: 0.0110
  vb_einziehen: 0.0087
  vb_naître: 0.0085
  vb_savoir: 0.0084


/var/folders/zf/kzq3m71s5m5_bwfn9npz8j7h0000gn/T/ipykernel_81151/2603001579.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, head, tag in zip(axes, [m._at, m._isat], ["at", "isAt"]):
    imp = head.clf.feature_importances_
    verb_mask = np.array([n.startswith("vb_") for n in feature_names])
    verb_mass = imp[verb_mask].sum()
    struct_mass = imp[~verb_mask].sum()

    ax.bar(["Structural", "Verb bag"], [struct_mass, verb_mass],
           color=["#e07b3a", "#4c8dc4"])
    ax.set_title(f"{tag}: importance by group")
    ax.set_ylabel("Total importance mass")
    print(f"{tag} — structural: {struct_mass:.4f}  verb_bag: {verb_mass:.4f}")

plt.tight_layout()
plt.savefig("/Users/mayadeneva/Documents/uni/ozt/hipe/notebooks/02_struct_vs_verb.png", dpi=120)
plt.show()
print("Saved 02_struct_vs_verb.png")

at — structural: 0.0187  verb_bag: 0.9813
isAt — structural: 0.0434  verb_bag: 0.9566
Saved 02_struct_vs_verb.png


/var/folders/zf/kzq3m71s5m5_bwfn9npz8j7h0000gn/T/ipykernel_81151/3885890636.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
import re

def is_ocr_junk(name):
    """Heuristic: contains a digit, non-alpha char, or length < 2."""
    stem = name[3:]  # strip 'vb_'
    return len(stem) < 2 or bool(re.search(r'[^a-zA-Z]', stem))

imp_at = m._at.clf.feature_importances_
verb_idx = np.where(np.array([n.startswith("vb_") for n in feature_names]))[0]
verb_imp = imp_at[verb_idx]
top25_verb_idx = verb_idx[np.argsort(verb_imp)[::-1][:25]]

print("Top 25 vb_* features for 'at' target (OCR junk flagged with [OCR]):")
print(f"{'Feature':<30} {'Importance':>12}  {'OCR?'}")
print("-" * 55)
for i in top25_verb_idx:
    flag = "[OCR]" if is_ocr_junk(feature_names[i]) else ""
    print(f"{feature_names[i]:<30} {imp_at[i]:>12.5f}  {flag}")

Top 25 vb_* features for 'at' target (OCR junk flagged with [OCR]):
Feature                          Importance  OCR?
-------------------------------------------------------
vb_want                             0.00581  
vb_reconnaître                      0.00525  [OCR]
vb_naître                           0.00477  [OCR]
vb_accompagner                      0.00477  
vb_know                             0.00460  
vb_55à                              0.00449  [OCR]
vb_habiter                          0.00437  
vb_parti¬                           0.00404  [OCR]
vb_welcome                          0.00402  
vb_arriver                          0.00383  
vb_note                             0.00383  
vb_mitgetheilen                     0.00383  
vb_remettre                         0.00374  
vb_machen                           0.00369  
vb_terminer                         0.00369  
vb_arir                             0.00368  
vb_montrer                          0.00366  
vb_geb.                 

## Findings

1. **`has_pres` (present tense) is the top `isAt` signal**, validating the hypothesis that present-tense sentences are the primary indicator of a current birth/residence location relationship.

2. **`dist_chars` (character distance between entities) matters** in both heads — proximity in text is a reliable proxy for relationship salience.

3. **Real, meaningful verbs surface** in the top-20 (e.g. `habiter`, `arriver`, `exister`, `naître`, `lived`) confirming the bag-of-lemmas captures genuine linguistic signals.

4. **BUT the verb bag is polluted by OCR-garbled tokens** (digits, non-alpha characters, ultra-short stems) and is fragmented across three languages (en/de/fr), which caps performance. Each real verb concept appears under three different lemma keys with diluted importance.

### Next steps
- Harder OCR filtering (require stem to be purely alphabetic and ≥ 3 chars)
- Replace per-language verb lemmas with multilingual verb/context **embeddings** — robust to OCR noise and naturally cross-lingual (e.g. mBERT/XLM-R sentence embeddings as features)